<a href="https://colab.research.google.com/github/Yohannis-10/AI-journey/blob/main/day_1_Report_Card.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
Report Card System
------------------
A complete console program that:
  - collects student marks with input validation
  - calculates total, average, grade and pass/fail result
  - prints a formatted report card
  - ranks the whole class
  - saves report cards to a text file

Works in Google Colab and in any normal Python 3.8+ environment.
"""

from dataclasses import dataclass, field

# ----------------------------------------------------------------------
# Settings (change these without touching the logic)
# ----------------------------------------------------------------------
SUBJECTS = ["Math", "English", "Physics", "Chemistry", "Biology"]
CLASS_NAME = "Grade 10"
MAX_MARK = 100
SUBJECT_PASS_MARK = 35      # a mark below this in any subject = FAIL
AVERAGE_PASS_MARK = 50      # an average below this = FAIL
LINE_WIDTH = 36

GRADE_SCALE = [             # checked from the highest to the lowest
    (90, "A"),
    (80, "B"),
    (70, "C"),
    (60, "D"),
]
LOWEST_GRADE = "F"


# ----------------------------------------------------------------------
# Data model
# ----------------------------------------------------------------------
@dataclass
class Student:
    name: str
    marks: list = field(default_factory=list)

    @property
    def total(self):
        return sum(self.marks)

    @property
    def average(self):
        return self.total / len(self.marks) if self.marks else 0.0


# ----------------------------------------------------------------------
# Calculation functions
# ----------------------------------------------------------------------
def get_grade(average):
    """Return the letter grade for an average mark."""
    for minimum, letter in GRADE_SCALE:
        if average >= minimum:
            return letter
    return LOWEST_GRADE


def get_result(student):
    """Return 'PASS' or 'FAIL' using the subject and average rules."""
    if any(mark < SUBJECT_PASS_MARK for mark in student.marks):
        return "FAIL"
    if student.average < AVERAGE_PASS_MARK:
        return "FAIL"
    return "PASS"


def rank_students(students):
    """Return students sorted from the highest to the lowest total."""
    return sorted(students, key=lambda s: s.total, reverse=True)


# ----------------------------------------------------------------------
# Input functions (with validation)
# ----------------------------------------------------------------------
def ask_text(prompt):
    """Ask for text and repeat until it is not empty."""
    while True:
        value = input(prompt).strip()
        if value:
            return value
        print("  This field cannot be empty.")


def ask_mark(subject):
    """Ask for one mark and repeat until it is a number from 0 to MAX_MARK."""
    while True:
        try:
            mark = float(input(f"  {subject} mark (0-{MAX_MARK}): "))
        except ValueError:
            print("  Please type a number.")
            continue
        if 0 <= mark <= MAX_MARK:
            return mark
        print(f"  Mark must be between 0 and {MAX_MARK}.")


def ask_yes_no(prompt):
    """Return True for yes and False for no."""
    while True:
        answer = input(prompt).strip().lower()
        if answer in ("y", "yes"):
            return True
        if answer in ("n", "no"):
            return False
        print("  Please type yes or no.")


def collect_student():
    """Ask for a student's name and all subject marks."""
    name = ask_text("Student name: ")
    marks = [ask_mark(subject) for subject in SUBJECTS]
    return Student(name=name, marks=marks)


# ----------------------------------------------------------------------
# Output functions
# ----------------------------------------------------------------------
def format_report_card(student):
    """Build the report card as text, so it can be printed or saved."""
    border = "=" * LINE_WIDTH
    divider = "-" * LINE_WIDTH
    best = max(student.marks)
    worst = min(student.marks)
    best_subject = SUBJECTS[student.marks.index(best)]
    worst_subject = SUBJECTS[student.marks.index(worst)]

    lines = [
        border,
        "REPORT CARD".center(LINE_WIDTH),
        border,
        f"Name : {student.name}",
        f"Class: {CLASS_NAME}",
        divider,
    ]
    lines.append(f"{'Subject':<16}{'Mark':>7}{'Grade':>6}{'Status':>7}".rstrip())
    lines.append(divider)
    for subject, mark in zip(SUBJECTS, student.marks):
        status = "FAIL" if mark < SUBJECT_PASS_MARK else "PASS"
        lines.append(f"{subject:<16}{mark:>7.1f}{get_grade(mark):>6}{status:>7}")
    lines += [
        divider,
        f"{'Total':<16}{student.total:>7.1f}",
        f"{'Average':<16}{student.average:>7.1f}{get_grade(student.average):>6}",
        f"{'Result':<16}{get_result(student):>7}",
        divider,
        f"Best subject  : {best_subject} ({best:.0f})",
        f"Weakest subject: {worst_subject} ({worst:.0f})",
        border,
    ]
    return "\n".join(lines)


def format_ranking(students):
    """Build the class ranking table as text."""
    lines = ["CLASS RANKING".center(LINE_WIDTH), "-" * LINE_WIDTH]
    for position, s in enumerate(rank_students(students), start=1):
        lines.append(f"{position:>2}. {s.name:<16}{s.average:>7.1f}  {get_grade(s.average)}")
    class_average = sum(s.average for s in students) / len(students)
    lines += ["-" * LINE_WIDTH, f"Class average: {class_average:.1f}"]
    return "\n".join(lines)


def save_report(students, filename="report_cards.txt"):
    """Save every report card and the ranking to a text file."""
    with open(filename, "w", encoding="utf-8") as file:
        for student in students:
            file.write(format_report_card(student) + "\n\n")
        file.write(format_ranking(students) + "\n")
    print(f"\nSaved to {filename}")


# ----------------------------------------------------------------------
# Main program
# ----------------------------------------------------------------------
def main():
    print("Welcome to the Report Card System\n")
    students = []

    while True:
        print(f"\n--- Student {len(students) + 1} ---")
        students.append(collect_student())
        if not ask_yes_no("\nAdd another student? (yes/no): "):
            break

    print()
    for student in rank_students(students):
        print(format_report_card(student))
        print()

    print(format_ranking(students))

    if ask_yes_no("\nSave report cards to a file? (yes/no): "):
        save_report(students)


if __name__ == "__main__":
    main()


Welcome to the Report Card System


--- Student 1 ---
Student name: Jaatoo Tolasaa
  Math mark (0-100): 93
  English mark (0-100): 86
  Physics mark (0-100): 98
  Chemistry mark (0-100): 96
  Biology mark (0-100): 94

Add another student? (yes/no): n

            REPORT CARD             
Name : Jaatoo Tolasaa
Class: Grade 10
------------------------------------
Subject            Mark Grade Status
------------------------------------
Math               93.0     A   PASS
English            86.0     B   PASS
Physics            98.0     A   PASS
Chemistry          96.0     A   PASS
Biology            94.0     A   PASS
------------------------------------
Total             467.0
Average            93.4     A
Result             PASS
------------------------------------
Best subject  : Physics (98)
Weakest subject: English (86)

           CLASS RANKING            
------------------------------------
 1. Jaatoo Tolasaa     93.4  A
------------------------------------
Class average: 93.4

Sa